# Étape 3 — Conception de l'auto-encodeur (`pill`)

TP B6, partie 2. Construire un auto-encodeur CNN (3 conv encodeur + 3 déconv
décodeur, sortie `sigmoid`), entraîné uniquement sur les pièces saines : ce qu'il
reconstruit mal devient suspect.

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from indusense.vision.model import build_autoencoder, compression_ratio

IMAGE_SIZE = (256, 256)
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Décision — taille du goulot d'étranglement

**Décision** : un goulot à **16 canaux** sur une grille spatiale 32×32 (après 3
convolutions stride-2 sur une entrée 256×256), soit un ratio de compression visé
d'environ **×12**.

**Pourquoi** : l'énoncé identifie un piège central — si le ratio de compression est
proche de 1, l'auto-encodeur peut apprendre une quasi-**identité** et reconstruire
aussi bien les défauts que les pièces saines, ce qui annule le signal d'anomalie
(l'erreur de reconstruction ne distingue plus rien). À l'inverse, un goulot trop
serré perd trop d'information et dégrade la reconstruction même des pièces saines.
On vise ici un ratio nettement supérieur à 1 (×12) pour donner à l'auto-encodeur une
réelle contrainte de compression, en pariant sur une meilleure séparation sain/défaut
— au prix d'un compromis à vérifier empiriquement dans les notebooks suivants
(l'AUROC de l'étape 6 tranchera si ce choix était le bon).

In [2]:
model = build_autoencoder(image_size=IMAGE_SIZE, latent_channels=16)
model.summary()

Model: "pill_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv1 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_conv2 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Conv2D)                 │ (None, 32, 32, 16)     │         9,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_deconv1 (Conv2DTranspose)   │ (None, 64, 64, 64)     │         9,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_deconv2 (Conv2DTranspose)   │ (None, 128, 128, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction                  │ (None, 256, 256, 3)    │           867 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,235 (223.57 KB)

 Trainable params: 57,235 (223.57 KB)

 Non-trainable params: 0 (0.00 B)

## Livrable — nombre de paramètres, goulot et ratio de compression

In [3]:
ratio = compression_ratio(model)
latent_shape = tuple(model.get_layer("latent").output.shape[1:])
n_input = IMAGE_SIZE[0] * IMAGE_SIZE[1] * 3
n_latent = latent_shape[0] * latent_shape[1] * latent_shape[2]
n_params = model.count_params()

lines = [
    f"- **Paramètres totaux** : {n_params:,} — modèle volontairement petit (227 images "
    f"saines d'entraînement, pas de quoi nourrir un réseau profond).",
    f"- **Entrée** : {IMAGE_SIZE[0]}×{IMAGE_SIZE[1]}×3 = {n_input:,} valeurs.",
    f"- **Goulot d'étranglement (`latent`)** : {latent_shape[0]}×{latent_shape[1]}×"
    f"{latent_shape[2]} = {n_latent:,} valeurs.",
    f"- **Ratio de compression** : entrée ÷ latent = ×{ratio:.1f}.",
]
display(Markdown(chr(10).join(lines)))

- **Paramètres totaux** : 57,235 — modèle volontairement petit (227 images saines d'entraînement, pas de quoi nourrir un réseau profond).
- **Entrée** : 256×256×3 = 196,608 valeurs.
- **Goulot d'étranglement (`latent`)** : 32×32×16 = 16,384 valeurs.
- **Ratio de compression** : entrée ÷ latent = ×12.0.

## Point de réflexion — le piège de la quasi-identité

Un ratio proche de **×1** signifierait que le goulot contient presque autant
d'information que l'image d'origine : l'auto-encodeur n'a alors aucune raison
d'apprendre une représentation compacte du « normal », il peut se contenter de
recopier l'entrée (à la précision des convolutions près) — y compris les défauts,
qui seraient alors reconstruits presque aussi bien que le reste. L'erreur de
reconstruction cesserait d'être un signal utile.

Avec **×12**, le modèle est forcé de résumer chaque image en une fraction de son
information — mais la grille latente reste spatialement large (32×32), donc une
partie non négligeable de la structure locale est préservée malgré tout. Ce n'est
donc pas une garantie absolue contre le piège, seulement un choix qui le rend moins
probable qu'un ratio proche de 1. Deux leviers restent disponibles si l'évaluation
(étape 6) montre une séparation sain/défaut insuffisante : resserrer encore le
goulot (moins de canaux latents), ou passer à une perte **SSIM** plus sensible aux
altérations de texture qu'une simple erreur pixel à pixel (MSE).